# HDAR Cross-Platform Continuation Proof — Google Colab Edition

This notebook runs the HDAR proof on **Google Colab** (Linux x86_64) as **Host B**.

| Role | Platform | Description |
|------|----------|-------------|
| Host A | macOS arm64 | Built and signed the deploy package |
| Host B | Google Colab (Linux x86_64) | Runs the continuation pipeline |
| Verifier | Colab (portability test) | Independently verifies with Host A platform = macOS |

## Expected Result
All 13 checks pass, including `platforms_differ` (macOS vs Linux).

## What This Proves
1. A workspace state capsule signed on macOS can be continued on Linux
2. The 5-stage deterministic pipeline produces identical output hash
3. Cryptographic lineage (E1→E2) is verified
4. Semantic correctness is independently recomputed (5 predicates)
5. Internal stage chain (Merkle-like) is intact
6. Evidence packet has independent Ed25519 signature
7. Platforms genuinely differ (cross-platform continuation)


In [ ]:
# Cell 1: Decode deploy package from embedded base64
import base64, os, tarfile, io

DEPLOY_B64 = "..."

os.makedirs("/content/hdar-demo", exist_ok=True)
raw = base64.b64decode(DEPLOY_B64)
with tarfile.open(fileobj=io.BytesIO(raw), mode="r:gz") as tf:
    tf.extractall("/content/hdar-demo")

print("Deploy package extracted to /content/hdar-demo/")
for f in sorted(os.listdir("/content/hdar-demo")):
    print(f"  {f}")

In [ ]:
# Cell 2: Install pinned dependencies
!pip install cryptography==44.0.1 -q
import cryptography
print(f"cryptography {cryptography.__version__} installed")

In [ ]:
# Cell 3: Verify runner integrity
import hashlib, json

with open("/content/hdar-demo/run_on_host_b.py", "rb") as f:
    runner_hash = hashlib.sha256(f.read()).hexdigest()

EXPECTED_RUNNER_SHA = ""

print(f"Runner SHA-256:    {runner_hash}")
print(f"Expected SHA-256:  {EXPECTED_RUNNER_SHA}")
if runner_hash == EXPECTED_RUNNER_SHA:
    print("PASS: Runner hash verified")
else:
    print("FAIL: RUNNER HASH MISMATCH")
    raise SystemExit(1)

In [ ]:
# Cell 4: Run Host B — 5-stage pipeline on Colab (Linux x86_64)
import subprocess, json, platform

print(f"Host B platform: {platform.platform()}")

OWNER_PUB = "3e24d007f4fec6b10befb59d2fe77fd53fb5e4dfef860bc96350bc83e475e199"

cmd = [
    "python3", "/content/hdar-demo/run_on_host_b.py",
    "--bundle", "transport_capsule_epoch_1_signed.tar.gz",
    "--host-a-report", "host_a_build_report.json",
    "--owner-public-key", OWNER_PUB,
    "--verify-runner-hash", EXPECTED_RUNNER_SHA,
    "--host-label", "google-colab-host-b",
    "--operator-identity", "google-colab-linux-x86_64",
    "--out", "host_b_output"
]

result = subprocess.run(cmd, capture_output=True, text=True, timeout=120,
                       cwd="/content/hdar-demo")
print(f"Exit code: {result.returncode}")
print("Last 800 chars of output:")
print(result.stdout[-800:])
if result.returncode != 0:
    print("STDERR:", result.stderr[-500:])
    raise SystemExit(1)

# Parse the report from stdout
report = json.loads(result.stdout)
tc = report.get("task_continuation", {})
print(f"\nTask: {tc.get('task')}")
print(f"Stages: {tc.get('stages_completed')}")
print(f"Output hash match: {tc.get('computed_output_hash') == tc.get('expected_output_hash')}")
print(f"Platforms differ: {report.get('host_b_platform', '') != 'macOS-26.5.2-arm64-arm-64bit-Mach-O'}")

In [ ]:
# Cell 5: Run the independent verifier (portability test on Colab)
import subprocess, json

# Host A platform is macOS arm64 (where deploy package was built)
HOST_A_PLATFORM = "macOS-26.5.2-arm64-arm-64bit-Mach-O"

cmd = [
    "python3", "/content/hdar-demo/third_party_verifier.py",
    "--capsule-e1", "host_b_output/capsule_epoch_1",
    "--capsule-e2", "host_b_output/capsule_epoch_2",
    "--host-b-report", "host_b_output/host_b_report.json",
    "--evidence-packet", "host_b_output/host_b_evidence_packet.json",
    "--owner-public-key", OWNER_PUB,
    "--host-a-platform", HOST_A_PLATFORM
]

result = subprocess.run(cmd, capture_output=True, text=True, timeout=60,
                       cwd="/content/hdar-demo")

try:
    verdict = json.loads(result.stdout)
    passed = verdict["passed"]
    total = verdict["total_checks"]
    all_ok = verdict["all_checks_passed"]
    print("=" * 60)
    print(f"  VERIFIER RESULT: {passed}/{total} checks passed")
    print(f"  ALL PASSED: {all_ok}")
    print("=" * 60)
    for c in verdict["checks"]:
        status = "PASS" if c["ok"] else "FAIL"
        print(f"  [{status}] {c['check']:30s} {c['reason'][:70]}")
    print()
    vb = verdict.get("version_binding", {})
    print(f"  Protocol:  {vb.get('protocol_version', '?')}")
    print(f"  Verifier:  {vb.get('verifier_version', '?')}")
    print(f"  Worker:    {vb.get('worker_version', '?')}")
    print(f"  Ruleset:   {vb.get('ruleset_version', '?')}")
    print(f"  Verifier SHA-256: {verdict.get('verifier_sha256', '?')[:32]}...")
except json.JSONDecodeError:
    print(f"Verifier exit code: {result.returncode}")
    print(f"STDOUT: {result.stdout[:2000]}")
    print(f"STDERR: {result.stderr[:2000]}")

In [ ]:
# Cell 6: Display lifecycle events from evidence packet
import json

with open("/content/hdar-demo/host_b_output/host_b_evidence_packet.json") as f:
    ep = json.load(f)

print("Lifecycle Events:")
print("-" * 80)
for event in ep.get("lifecycle_events", []):
    print(f"  {event['event']:25s}  {event['timestamp']}  {event['detail'][:50]}")

print(f"\nPost-sign modifications: {ep.get('lifecycle_events_post_sign', False)}")
print(f"Evidence packet SHA-256: {ep.get('evidence_packet_sha256', '?')[:32]}...")
print(f"Host B public key:       {ep.get('host_b_public_key', '?')[:32]}...")
print(f"Signature algorithm:     {ep.get('signature_algorithm', '?')}")

In [ ]:
# Cell 7: Display semantic correctness details
import json

# Re-read verifier output from cell 5
verdict = json.loads(result.stdout)
sem = None
for c in verdict["checks"]:
    if c["check"] == "semantic_correctness":
        sem = c
        break

if sem:
    print(f"Semantic Correctness: {'PASS' if sem['ok'] else 'FAIL'}")
    print(f"Predicates checked: {sem.get('predicates_checked', 0)}")
    print(f"Reason: {sem['reason']}")
    ic = sem.get("independently_computed", {})
    print(f"\nIndependently computed:")
    print(f"  Total records:    {ic.get('total_records', '?')}")
    print(f"  Valid records:    {ic.get('valid_records', '?')}")
    print(f"  Rejected records: {ic.get('rejected_records', '?')}")
    print(f"  Rejected IDs:     {ic.get('rejected_ids', [])}")
    print(f"  Categories:       {ic.get('categories', [])}")
    print(f"  Category sums:    {ic.get('category_sums', {})}")
    print(f"  Category counts:  {ic.get('category_counts', {})}")
    print(f"  Tier counts:      {ic.get('tier_counts', {})}")

## Run Verifier on Host A (Authoritative)

After running this notebook, download the results and run the verifier on your Mac (Host A)
to get the authoritative post-destruction verdict:

1. Run the download cell below to get `hdar-colab-results.tar.gz`
2. On your Mac:
```bash
tar xzf hdar-colab-results.tar.gz
python3 deploy-package/third_party_verifier.py \
  --capsule-e1 capsule_epoch_1 \
  --capsule-e2 host_b_output/capsule_epoch_2 \
  --host-b-report host_b_output/host_b_report.json \
  --evidence-packet host_b_output/host_b_evidence_packet.json \
  --owner-public-key $(cat deploy-package/owner_public_key.txt) \
  --host-a-platform "$(python3 -c 'import platform; print(platform.platform())')"
```

This gives the authoritative verdict with the verifier running on a different machine
from where Host B executed.

In [ ]:
# Cell 8: Download results for Host A verification
from google.colab import files
import tarfile, os

with tarfile.open("/content/hdar-colab-results.tar.gz", "w:gz") as tf:
    tf.add("/content/hdar-demo/host_b_output", arcname="host_b_output")
    tf.add("/content/hdar-demo/host_b_output/capsule_epoch_1", arcname="capsule_epoch_1")

print(f"Results archive: {os.path.getsize("/content/hdar-colab-results.tar.gz")} bytes")
files.download("/content/hdar-colab-results.tar.gz")
print("\nDownload started. Run the verifier on your Mac (Host A) for authoritative verdict.")